### CNES Hospitais/Leitos UTI

Bronze -> Silver: Hospitais e Leitos CNES, competencias 2020–2024.
Instalar: %pip install pandas pyarrow
Executar: %run tratar_cnes_leitos_silver.py
Nao baixa dados nem conecta ao PostgreSQL. Nao altera a Bronze.
Uma pasta de execucao nova evita misturar resultados de rodadas diferentes.
Quantidade vazia vira NULL, nunca zero. Duplicidades nao sao removidas.

In [5]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import re
import unicodedata
import uuid
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [6]:
PROJETO = Path(r'D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil')
BRONZE = PROJETO / 'Camada_Bronze'
SILVER = PROJETO / 'Camada_Silver'
# Se houver duas coletas CNES na Bronze, indique a pasta parquet escolhida aqui.
ENTRADA = None
ANOS = tuple(range(2020,2025))
LOTE = 20000
UFS = set('AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI RJ RN RS RO RR SC SP SE TO'.split())
QUANTIDADES = ['leitos_existentes','leitos_sus'] + [f'uti_{tipo}_{sufixo}' for tipo in ('total','adulto','pediatrico','neonatal','queimado','coronariana') for sufixo in ('exist','sus')]

In [7]:
# Dependencia: executar tratar_populacao_ibge_silver.py antes deste script.
# None procura UMA unica execucao IBGE. Se houver varias, indique o Parquet revisado.
IBGE_PARQUET = None
VERSAO_EQUIVALENCIAS = '2026-10-27.v1'
COLUNAS_VINCULO = ['codigo_ibge_7','municipio_ibge','metodo_vinculo','status_vinculo']
# Tabela versionada junto ao codigo e exportada em cada execucao.
# Nome CNES, UF e codigo foram conferidos no diagnostico do projeto.
# Nao aplicar fora de 2020–2024 nem usar aproximacao automatica de nomes.
EQUIVALENCIAS = [
    ('BA','SANTA TERESINHA','2928505'),
    ('CE','ITAPAGE','2306306'),
    ('MG','BRASOPOLIS','3108909'),
    ('MT','POXOREO','5107008'),
    ('MT','SANTO ANTONIO DO LEVERGER','5107800'),
    ('PA','ELDORADO DOS CARAJAS','1502954'),
    ('PB','SERIDO','2515401'),
    ('PE','BELEM DE SAO FRANCISCO','2601607'),
    ('PE','IGUARACI','2606903'),
    ('PE','LAGOA DO ITAENGA','2608503'),
    ('RJ','PARATI','3303807'),
    ('RJ','TRAJANO DE MORAIS','3305901'),
    ('RN','ACU','2400208'),
    ('RN','ARES','2401206'),
    ('RN','AUGUSTO SEVERO','2401305'),
    ('RN','PRESIDENTE JUSCELINO','2410306'),
    ('RR','SAO LUIZ','1400605'),
    ('SP','MOJI MIRIM','3530805'),
    ('SP','SAO LUIS DO PARAITINGA','3550001'),
]
FONTES_HISTORICAS = {
    ('PB','SERIDO'): 'https://www.ibge.gov.br/biblioteca/visualizacao/dtb/paraiba/saovicentedoserido.pdf',
    ('RN','AUGUSTO SEVERO'): 'https://www.cmcampogrande.rn.gov.br/cidade',
    ('RN','PRESIDENTE JUSCELINO'): 'https://educa.ibge.gov.br/criancas/voce-sabia/22421-mudanca-de-nome.html',
}


def chave_municipio(valor):
    if pd.isna(valor):
        return ''
    texto = unicodedata.normalize('NFKD', str(valor)).encode('ascii','ignore').decode().upper()
    return re.sub('[^A-Z0-9]', '', texto)


def sha256_arquivo(p):
    import hashlib
    h = hashlib.sha256()
    with p.open('rb') as f:
        for bloco in iter(lambda: f.read(1024*1024), b''):
            h.update(bloco)
    return h.hexdigest()


def preparar_ibge():
    if IBGE_PARQUET is None:
        encontrados = sorted((SILVER/'dados'/'populacao_ibge').rglob('populacao_municipios_2020_2024.parquet'))
        if len(encontrados) != 1:
            raise ValueError(f'Encontrei {len(encontrados)} Parquets IBGE na Silver. '
                             f'Defina IBGE_PARQUET com a execucao revisada. Arquivos: {encontrados}')
        caminho = encontrados[0]
    else:
        caminho = Path(IBGE_PARQUET)
    if not caminho.is_file():
        raise FileNotFoundError(caminho)
    print(f'IBGE de referencia: {caminho}', flush=True)
    ibge = pd.read_parquet(caminho)
    obrigatorias = {'ano_analise','uf','municipio','codigo_ibge_7','populacao'}
    if not obrigatorias.issubset(ibge.columns):
        raise ValueError(f'Faltam colunas no IBGE: {obrigatorias-set(ibge.columns)}')
    for c in ('uf','municipio','codigo_ibge_7'):
        ibge[c] = ibge[c].astype('string').str.strip().replace('',pd.NA)
    ibge['uf'] = ibge.uf.str.upper()
    anos = pd.to_numeric(ibge.ano_analise, errors='coerce')
    if (anos.isna() | ~anos.isin(ANOS)).any() or set(anos) != set(ANOS):
        raise ValueError('IBGE deve conter apenas os cinco anos de 2020 a 2024, completos.')
    ibge['ano_analise'] = anos.astype(int)
    if (~ibge.codigo_ibge_7.str.fullmatch('[0-9]{7}',na=False)).any() or (~ibge.uf.isin(UFS)).any() or ibge.municipio.isna().any():
        raise ValueError('Chaves ou nomes invalidos no IBGE Silver.')
    if ibge.duplicated(['ano_analise','codigo_ibge_7']).any():
        raise ValueError('IBGE duplicado por codigo/ano. Nenhum vinculo sera realizado.')
    pop = pd.to_numeric(ibge.populacao, errors='coerce')
    if (ibge.populacao.notna() & (pop.isna() | pop.le(0) | pop.mod(1).ne(0))).any():
        raise ValueError('Populacao invalida no IBGE Silver.')
    ibge['populacao'] = pop
    ibge['chave_nome'] = ibge.municipio.map(chave_municipio)
    por_codigo = {}
    por_nome = {}
    for r in ibge.to_dict('records'):
        por_codigo[(r['ano_analise'],r['uf'],r['codigo_ibge_7'])] = r
        if r['chave_nome']:
            por_nome.setdefault((r['ano_analise'],r['uf'],r['chave_nome']),[]).append(r)
    regras = {}
    tabela = []
    for uf,nome,codigo in EQUIVALENCIAS:
        chave = (uf,chave_municipio(nome))
        if chave in regras:
            raise ValueError(f'Equivalencia duplicada: {chave}')
        regras[chave] = codigo
        fonte = FONTES_HISTORICAS.get((uf,nome))
        tabela.append(dict(versao=VERSAO_EQUIVALENCIAS,uf=uf,nome_cnes=nome,
            chave_nome=chave[1],codigo_ibge_7=codigo,ano_inicio=2020,ano_fim=2024,
            justificativa='Denominacao historica' if fonte else 'Variante de grafia/denominacao conferida no diagnostico CNES x IBGE do projeto',
            fonte=fonte or 'Resultados SQL fornecidos pelo usuario; conferir contra o IBGE Silver selecionado'))
    ambiguos = ibge.loc[ibge.duplicated(['ano_analise','uf','chave_nome'],keep=False)].copy()
    return caminho, por_codigo, por_nome, regras, pd.DataFrame(tabela), ambiguos


def vincular_municipios(df, por_codigo, por_nome, regras):
    registros = []
    for ano,uf,nome in df[['ano','uf','municipio']].itertuples(index=False,name=None):
        chave = chave_municipio(nome)
        codigo = regras.get((uf,chave)) if int(ano) in ANOS else None
        candidatos = por_nome.get((int(ano),uf,chave),[]) if chave else []
        alvo = None
        metodo = 'NAO_VINCULADO'
        status = 'SEM_CORRESPONDENCIA'
        if codigo is not None:
            metodo = 'EQUIVALENCIA_EXPLICITA'
            alvo = por_codigo.get((int(ano),uf,codigo))
            if alvo is None:
                status = 'CODIGO_EQUIVALENCIA_AUSENTE_NO_ANO_UF'
            elif candidatos and any(r['codigo_ibge_7'] != codigo for r in candidatos):
                alvo = None
                status = 'CONFLITO_EQUIVALENCIA_NOME'
        elif len(candidatos) == 1:
            alvo = candidatos[0]
            metodo = 'NOME_NORMALIZADO'
        elif len(candidatos) > 1:
            status = 'CORRESPONDENCIA_AMBIGUA'
        if alvo is not None:
            status = 'VINCULADO_SEM_POPULACAO' if pd.isna(alvo['populacao']) else 'VINCULADO_COM_POPULACAO'
        registros.append((alvo['codigo_ibge_7'] if alvo else None,
                          alvo['municipio'] if alvo else None,metodo,status))
    saida = df.copy()
    for i,c in enumerate(COLUNAS_VINCULO):
        saida[c] = pd.array([r[i] for r in registros],dtype='string')
    if len(saida) != len(df):
        raise RuntimeError('Vinculo alterou a quantidade de linhas.')
    return saida


def normalizar(nome):
    x = unicodedata.normalize('NFKD',nome).encode('ascii','ignore').decode().lower()
    x = re.sub('[^a-z0-9]+','_',x).strip('_')
    return {'comp':'competencia','leitos_existente':'leitos_existentes'}.get(x,x)


def encontrar():
    if not BRONZE.is_dir(): raise FileNotFoundError(f'Bronze nao encontrada: {BRONZE}')
    raiz = Path(ENTRADA) if ENTRADA is not None else BRONZE
    resultado=[]
    for ano in ANOS:
        encontrados=sorted(raiz.rglob(f'cnes_leitos_{ano}.parquet'))
        if len(encontrados)!=1:
            raise ValueError(f'{ano}: esperava um Parquet CNES, encontrei {len(encontrados)} em {raiz}. Ajuste ENTRADA para uma unica coleta. Arquivos: {encontrados}')
        resultado.append((ano,encontrados[0]))
    return resultado


def esquema_entrada(arquivos):
    colunas=[];mapeamentos={}
    reservadas={'data_competencia','ano','mes','arquivo_origem','tratado_em_utc','motivos_rejeicao','fl_inconsistencia_leitos',*COLUNAS_VINCULO}
    for ano,p in arquivos:
        nomes=pq.read_schema(p).names;novos=[normalizar(c) for c in nomes]
        if len(set(novos))!=len(novos) or any(not re.fullmatch('[a-z_][a-z0-9_]{0,62}',c) for c in novos):
            raise ValueError(f'Nomes de colunas ambiguos/invalidos: {p}')
        if reservadas & set(novos): raise ValueError(f'Coluna reservada ja existe em {p}')
        obrigatorias={'cnes','competencia','uf','municipio',*QUANTIDADES}
        if not obrigatorias.issubset(novos): raise ValueError(f'Faltam colunas em {p}: {obrigatorias-set(novos)}')
        mapeamentos[str(p)]=dict(zip(nomes,novos))
        colunas.extend(c for c in novos if c not in colunas)
    return colunas,mapeamentos


def tratar(df,ano):
    df=df.copy()
    for c in df:
        df[c]=df[c].astype('string').str.strip().replace('',pd.NA)
    # Valores originais sao preservados na quarentena antes de qualquer conversao.
    bruto=df.copy();motivos=pd.Series('',index=df.index,dtype='string')
    def marcar(mask,texto):
        mask=mask.fillna(False)
        motivos.loc[mask]=motivos.loc[mask]+texto+';'
    # Corrige apenas codigos numericos inteiros com ate sete digitos.
    cnes=df['cnes'].str.replace(r'\.0+$','',regex=True)
    cnes_valido=cnes.str.fullmatch(r'\d{1,7}',na=False)
    marcar(~cnes_valido,'cnes_invalido_ou_ausente')
    df['cnes']=cnes.where(cnes_valido).str.zfill(7)
    cnes_corrigidos=int((df['cnes'].ne(bruto['cnes']) & cnes_valido).fillna(False).sum())
    comp=df['competencia'];comp_ok=comp.str.fullmatch(r'\d{4}(0[1-9]|1[0-2])',na=False)
    marcar(~comp_ok,'competencia_invalida_ou_ausente')
    marcar(comp_ok & comp.str[:4].ne(str(ano)),'competencia_diverge_ano_arquivo')
    df['uf']=df['uf'].str.upper();marcar(~df['uf'].isin(UFS),'uf_invalida_ou_ausente')
    for c in QUANTIDADES:
        s=df[c];numero=pd.to_numeric(s,errors='coerce')
        valido=numero.notna() & numero.ge(0) & numero.le(2147483647) & numero.mod(1).eq(0)
        marcar(s.notna() & ~valido,c+'_invalido')
        df[c]=numero.where(valido).astype('Int64')
    df['fl_inconsistencia_leitos']=df['leitos_sus'].gt(df['leitos_existentes']).astype('boolean')
    datas=pd.to_datetime(comp.where(comp_ok),format='%Y%m',errors='coerce')
    df['data_competencia']=datas.dt.date
    df['ano']=pd.to_numeric(comp.str[:4].where(comp_ok),errors='coerce').astype('Int64')
    df['mes']=pd.to_numeric(comp.str[4:].where(comp_ok),errors='coerce').astype('Int64')
    rejeitados=bruto.loc[motivos.ne('')].copy()
    rejeitados['motivos_rejeicao']=motivos.loc[rejeitados.index].str.rstrip(';')
    return df.loc[motivos.eq('')].copy(),rejeitados,cnes_corrigidos


def main():
    arquivos=encontrar();colunas,mapas=esquema_entrada(arquivos)
    ibge_path,por_codigo,por_nome,regras_geo,tabela_geo,ambiguos_geo=preparar_ibge()
    hash_ibge=sha256_arquivo(ibge_path)
    run=datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
    saida=SILVER/'dados'/'cnes_leitos'/run
    qualidade=SILVER/'qualidade'/'cnes_leitos'/run
    docs=SILVER/'documentacao'/'cnes_leitos'/run
    for p in (saida,qualidade,docs):p.mkdir(parents=True,exist_ok=False)
    momento=datetime.now(timezone.utc).isoformat()
    campos=[pa.field(c,pa.int64() if c in QUANTIDADES else pa.string()) for c in colunas]
    campos += [pa.field('fl_inconsistencia_leitos',pa.bool_()),pa.field('data_competencia',pa.date32()),pa.field('ano',pa.int16()),pa.field('mes',pa.int8()),pa.field('arquivo_origem',pa.string()),pa.field('tratado_em_utc',pa.string())]
    campos += [pa.field(c,pa.string()) for c in COLUNAS_VINCULO]
    esquema=pa.schema(campos)
    schema_q=pa.schema([(c,pa.string()) for c in colunas+['motivos_rejeicao','arquivo_origem']])
    final=saida/'cnes_leitos_2020_2024.parquet';tmp=final.with_suffix('.parquet.part')
    qfinal=qualidade/'registros_rejeitados.parquet';qtmp=qfinal.with_suffix('.parquet.part')
    chaves=Counter();meses=Counter();resumo=[];motivos_contagem=Counter();avisos=Counter();nulos=Counter()
    vinculos=Counter();metodos=Counter();auditoria_geo=Counter()
    with pq.ParquetWriter(tmp,esquema,compression='snappy') as w, pq.ParquetWriter(qtmp,schema_q,compression='snappy') as qw:
        for ano,p in arquivos:
            lidos=aceitos=rejeitados=corrigidos=0
            origem=str(p.relative_to(BRONZE)) if p.is_relative_to(BRONZE) else str(p)
            print(f'Tratando {ano}: {p.name}',flush=True)
            for batch in pq.ParquetFile(p).iter_batches(batch_size=LOTE):
                df=batch.to_pandas().rename(columns=mapas[str(p)]).reindex(columns=colunas)
                ok,ruins,n=tratar(df,ano);lidos+=len(df);aceitos+=len(ok);rejeitados+=len(ruins);corrigidos+=n
                ok=vincular_municipios(ok,por_codigo,por_nome,regras_geo)
                vinculos.update(ok['status_vinculo']);metodos.update(ok['metodo_vinculo'])
                for r in ok[['ano','uf','municipio',*COLUNAS_VINCULO]].itertuples(index=False,name=None):
                    auditoria_geo.update([tuple(None if pd.isna(v) else v for v in r)])
                ok['arquivo_origem']=origem;ok['tratado_em_utc']=momento
                ruins['arquivo_origem']=origem
                if len(ok):
                    w.write_table(pa.Table.from_pandas(ok,schema=esquema,preserve_index=False,safe=True))
                    chaves.update(zip(ok['cnes'],ok['competencia']));meses.update(ok['competencia'])
                    nulos.update({c:int(ok[c].isna().sum()) for c in QUANTIDADES})
                    for tipo in ['leitos']+[f'uti_{t}' for t in ('total','adulto','pediatrico','neonatal','queimado','coronariana')]:
                        exist='leitos_existentes' if tipo=='leitos' else tipo+'_exist'
                        avisos[tipo+'_sus_maior_existente']+=int(ok[tipo+'_sus'].gt(ok[exist]).fillna(False).sum())
                if len(ruins):
                    qw.write_table(pa.Table.from_pandas(ruins,schema=schema_q,preserve_index=False))
                    for s in ruins['motivos_rejeicao']:motivos_contagem.update(s.split(';'))
            if lidos!=pq.read_metadata(p).num_rows or lidos!=aceitos+rejeitados:raise RuntimeError(f'Contagens divergentes: {p}')
            resumo.append({'ano_arquivo':ano,'lidos':lidos,'aceitos':aceitos,'rejeitados':rejeitados,'cnes_padronizados_incluindo_quarentena':corrigidos})
    total=sum(x['aceitos'] for x in resumo);total_q=sum(x['rejeitados'] for x in resumo)
    if pq.read_metadata(tmp).num_rows!=total or pq.read_metadata(qtmp).num_rows!=total_q:raise RuntimeError('Contagens finais divergentes.')
    tmp.replace(final);qtmp.replace(qfinal)
    duplicadas=[{'cnes':k[0],'competencia':k[1],'registros':n} for k,n in chaves.items() if n>1]
    pd.DataFrame(duplicadas,columns=['cnes','competencia','registros']).to_csv(qualidade/'chaves_duplicadas.csv',sep=';',index=False,encoding='utf-8-sig')
    pd.DataFrame(resumo).to_csv(qualidade/'contagens_por_ano.csv',sep=';',index=False,encoding='utf-8-sig')
    esperados={f'{a}{m:02}' for a in ANOS for m in range(1,13)}
    faltantes=sorted(esperados-set(meses))
    geo_cols=['ano','uf','municipio',*COLUNAS_VINCULO,'registros_cnes']
    auditoria=pd.DataFrame([list(k)+[n] for k,n in auditoria_geo.items()],columns=geo_cols)
    auditoria.to_csv(qualidade/'auditoria_vinculos_municipais.csv',sep=';',index=False,encoding='utf-8-sig')
    pendentes=auditoria.loc[~auditoria.status_vinculo.isin(['VINCULADO_COM_POPULACAO','VINCULADO_SEM_POPULACAO'])]
    pendentes.to_csv(qualidade/'pendencias_vinculos_municipais.csv',sep=';',index=False,encoding='utf-8-sig')
    ambiguos_geo.to_csv(qualidade/'nomes_ambiguos_ibge.csv',sep=';',index=False,encoding='utf-8-sig')
    tabela_geo.to_csv(docs/'equivalencias_municipais.csv',sep=';',index=False,encoding='utf-8-sig')
    total_pendentes=int(pendentes.registros_cnes.sum())
    if sum(vinculos.values()) != total: raise RuntimeError('Contagem dos vinculos diverge da Silver.')
    revisao=bool(total_pendentes) or not total or bool(total_q or duplicadas or faltantes or any(v for k,v in avisos.items() if k!='leitos_sus_maior_existente'))
    status=('REVISAR_ANTES_DA_CARGA' if revisao else
            'VALIDADO_COM_RESSALVAS' if avisos['leitos_sus_maior_existente'] or vinculos['VINCULADO_SEM_POPULACAO'] or metodos['EQUIVALENCIA_EXPLICITA'] else
            'VALIDADO_NOS_TESTES_IMPLEMENTADOS')
    relatorio={'status':status,
        'versao_equivalencias':VERSAO_EQUIVALENCIAS,
        'arquivo_ibge_silver':str(ibge_path),'sha256_ibge_silver':hash_ibge,
        'status_vinculos_municipais':dict(vinculos),'metodos_vinculo':dict(metodos),
        'registros_com_vinculo_pendente':total_pendentes,
        'observacao_vinculo':'Cobertura nao comprova identidade geografica. Regras explicitas versionadas; sem fuzzy matching. Municipio original preservado.',
        'politica_ressalvas':'SUS maior que existente em leitos gerais: preservar valores e sinalizar. Nao representa correcao nem validacao dos valores da fonte.',
        'registros_sinalizados_leitos':avisos['leitos_sus_maior_existente'],
        'registros_silver':total,'registros_quarentena':total_q,'chaves_duplicadas':len(duplicadas),
        'registros_excedentes_por_chave':sum(x['registros']-1 for x in duplicadas),
        'competencias_ausentes':faltantes,'registros_por_competencia':dict(sorted(meses.items())),
        'motivos_rejeicao':dict(motivos_contagem),'avisos_preservados_na_silver':dict(avisos),
        'quantidades_nulas_na_silver':dict(nulos),'arquivos_entrada':[str(p) for _,p in arquivos],
        'parquet_silver':str(final),'observacao':'Nenhuma duplicata removida. Competencia ausente nao equivale a zero leitos.'}
    (qualidade/'relatorio_qualidade.json').write_text(json.dumps(relatorio,ensure_ascii=False,indent=2),encoding='utf-8')
    (docs/'mapeamento_colunas.json').write_text(json.dumps(mapas,ensure_ascii=False,indent=2),encoding='utf-8')
    sqltipos=lambda t:'BOOLEAN' if pa.types.is_boolean(t) else 'BIGINT' if pa.types.is_int64(t) else ('SMALLINT' if pa.types.is_integer(t) else ('DATE' if pa.types.is_date(t) else 'TEXT'))
    definicoes=[f'    "{f.name}" {sqltipos(f.type)}' for f in esquema]
    ddl='-- Revisar relatorio de qualidade antes da carga. Nao executado pelo script.\nCREATE SCHEMA IF NOT EXISTS silver;\nCREATE TABLE IF NOT EXISTS silver.cnes_leitos (\n'+',\n'.join(definicoes)+'\n);\n-- CNES + competencia: chave candidata; criar UNIQUE somente apos resolver duplicidades.\n'
    (docs/'criar_tabela_cnes_leitos.sql').write_text(ddl,encoding='utf-8')
    regras='''Vinculo municipal realizado na Silver com o IBGE Silver escolhido explicitamente ou unico encontrado.
Municipio CNES preservado. Codigo IBGE e municipio_ibge adicionados por ano e UF.
Normalizacao NFKD: remove acentos, espacos e pontuacao somente na chave de comparacao.
Equivalencias explicitas possuem prioridade, mas conflito com nome normalizado impede o vinculo.
Codigo inexistente no ano/UF, nome ambiguo ou sem correspondencia: preserva linha, codigo NULL e status de revisao.
Populacao nao e copiada ao CNES. VINCULADO_SEM_POPULACAO preserva codigo valido e gera ressalva.
Auditoria e CSV das equivalencias acompanham cada rodada; conservar codigo e CSV no Git.
O SQL gerado nao migra tabela existente. Carga anterior deve ser atualizada em etapa separada.
CNES hospitais e leitos: recorte 2020–2024. Grao esperado: estabelecimento (CNES) por competencia mensal.
Colunas em snake_case; strings aparadas; vazios -> NULL; quantidades inteiras nao negativas.
CNES numerico de 1 a 7 digitos e completado com zeros; sufixo .0 e retirado quando inteiro.
Data_competencia representa o primeiro dia do mes, nao a data real de um evento.
CNES, competencia, UF ou quantidades invalidas: linha completa separada em quarentena.
SUS maior que existente: alerta, sem corrigir valores ou eliminar a linha automaticamente.
fl_inconsistencia_leitos: TRUE se leitos_sus > leitos_existentes, FALSE se comparaveis sem divergencia, NULL se alguma quantidade ausente.
Leitos gerais divergentes permitem VALIDADO_COM_RESSALVAS quando as demais verificacoes passam.
Essa politica preserva os valores e nao comprova sua exatidao; revisar a contagem em cada execucao.
Divergencias em UTI, quarentena, duplicatas, meses ausentes ou base vazia continuam exigindo revisao.
Quantidade ausente continua NULL. Duplicatas por CNES/competencia sao mantidas e reportadas.
Nao soma leitos entre meses; nao soma UTI total com seus subtipos; nao mede ocupacao.
O SQL e apenas uma definicao inicial. Nao conecta nem carrega PostgreSQL.
Cada execucao fica isolada: carregar somente o Parquet da execucao revisada, sem acumular rodadas.
'''
    (docs/'regras_tratamento.txt').write_text(regras,encoding='utf-8')
    print('\nStatus: '+relatorio['status']);print('Dados: '+str(final));print('Qualidade: '+str(qualidade));print('SQL: '+str(docs/'criar_tabela_cnes_leitos.sql'))
    print('Vinculos municipais: '+json.dumps(dict(vinculos),ensure_ascii=False))
    print('Pendencias de vinculo: '+str(total_pendentes))
    return relatorio


if __name__=='__main__':
    resultado_cnes_silver=main()


IBGE de referencia: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Silver\dados\populacao_ibge\20261024_163052_e1db3d2b\populacao_municipios_2020_2024.parquet
Tratando 2020: cnes_leitos_2020.parquet
Tratando 2021: cnes_leitos_2021.parquet
Tratando 2022: cnes_leitos_2022.parquet
Tratando 2023: cnes_leitos_2023.parquet
Tratando 2024: cnes_leitos_2024.parquet

Status: VALIDADO_COM_RESSALVAS
Dados: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Silver\dados\cnes_leitos\20261027_153722_858fee3c\cnes_leitos_2020_2024.parquet
Qualidade: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Silver\qualidade\cnes_leitos\20261027_153722_858fee3c
SQL: D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil\Camada_Silver\documentacao\cnes_leitos\20261027_153722_858fee3c\criar_tabela_cnes_leitos.sql
Vinculos municipais: {"VINCULADO_COM_POPULACAO": 424204}
Pendencias de vinculo: 0


### Banco de dados PostgreSQL

In [8]:
from pathlib import Path
from getpass import getpass

import pyarrow.parquet as pq
import psycopg
from psycopg import sql

In [9]:
ARQUIVO = Path(
    r"D:\Cleber\GitHub\Projetos\observatorio_covid19_brasil"
    r"\Camada_Silver\dados\cnes_leitos"
    r"\20261027_153722_858fee3c"
    r"\cnes_leitos_2020_2024.parquet"
)


def atualizar_vinculos_cnes():
    colunas = [
        "cnes", "competencia", "ano", "uf",
        "codigo_ibge_7", "municipio_ibge",
        "metodo_vinculo", "status_vinculo",
    ]

    dados = pq.read_table(ARQUIVO, columns=colunas)

    if dados.num_rows == 0:
        raise ValueError("O Parquet está vazio.")

    senha = getpass("Senha do PostgreSQL: ")

    try:
        conexao = psycopg.connect(
            host="localhost",
            port=5432,
            dbname="observatorio_covid19_brasil",
            user="postgres",
            password=senha,
            connect_timeout=15,
        )
    finally:
        del senha

    # Todas as alterações são confirmadas juntas ao final.
    # Se alguma verificação falhar, são desfeitas.
    with conexao:
        with conexao.cursor() as cursor:
            cursor.execute("SET LOCAL lock_timeout = '15s'")

            cursor.execute("""
                LOCK TABLE silver.cnes_leitos
                IN SHARE ROW EXCLUSIVE MODE
            """)

            cursor.execute("""
                LOCK TABLE silver.populacao_ibge
                IN SHARE MODE
            """)

            cursor.execute("""
                CREATE TEMP TABLE tmp_vinculos_cnes (
                    cnes TEXT NOT NULL,
                    competencia TEXT NOT NULL,
                    ano SMALLINT NOT NULL,
                    uf TEXT NOT NULL,
                    codigo_ibge_7 TEXT NOT NULL,
                    municipio_ibge TEXT NOT NULL,
                    metodo_vinculo TEXT NOT NULL,
                    status_vinculo TEXT NOT NULL,
                    PRIMARY KEY (cnes, competencia)
                ) ON COMMIT DROP
            """)

            with cursor.copy("""
                COPY tmp_vinculos_cnes (
                    cnes, competencia, ano, uf,
                    codigo_ibge_7, municipio_ibge,
                    metodo_vinculo, status_vinculo
                ) FROM STDIN
            """) as copia:
                for lote in dados.to_batches(max_chunksize=20000):
                    for registro in lote.to_pylist():
                        copia.write_row(
                            tuple(registro[c] for c in colunas)
                        )

            cursor.execute("ANALYZE tmp_vinculos_cnes")

            # Confere se o Parquet e a tabela possuem as mesmas chaves.
            cursor.execute("""
                SELECT COUNT(*)
                FROM silver.cnes_leitos AS c
                FULL JOIN tmp_vinculos_cnes AS t
                    ON t.cnes = c.cnes
                   AND t.competencia = c.competencia
                WHERE c.cnes IS NULL
                   OR t.cnes IS NULL
                   OR c.ano IS DISTINCT FROM t.ano
                   OR c.uf IS DISTINCT FROM t.uf
            """)

            if cursor.fetchone()[0]:
                raise RuntimeError(
                    "Há diferenças de chaves, ano ou UF entre "
                    "o Parquet e a tabela. Atualização cancelada."
                )

            # Confere os vínculos contra o IBGE carregado no banco.
            cursor.execute("""
                SELECT COUNT(*)
                FROM tmp_vinculos_cnes AS t
                LEFT JOIN silver.populacao_ibge AS p
                    ON p.ano_analise = t.ano
                   AND p.codigo_ibge_7 = t.codigo_ibge_7
                   AND p.uf = t.uf
                WHERE p.codigo_ibge_7 IS NULL
                   OR p.municipio IS DISTINCT FROM t.municipio_ibge
                   OR t.metodo_vinculo NOT IN (
                       'NOME_NORMALIZADO',
                       'EQUIVALENCIA_EXPLICITA'
                   )
                   OR t.status_vinculo IS DISTINCT FROM (
                       CASE
                           WHEN p.populacao IS NULL
                               THEN 'VINCULADO_SEM_POPULACAO'
                           ELSE 'VINCULADO_COM_POPULACAO'
                       END
                   )
            """)

            if cursor.fetchone()[0]:
                raise RuntimeError(
                    "Os vínculos divergem do IBGE no banco. "
                    "Atualização cancelada."
                )

            cursor.execute("""
                UPDATE silver.cnes_leitos AS c
                SET
                    codigo_ibge_7 = t.codigo_ibge_7,
                    municipio_ibge = t.municipio_ibge,
                    metodo_vinculo = t.metodo_vinculo,
                    status_vinculo = t.status_vinculo
                FROM tmp_vinculos_cnes AS t
                WHERE c.cnes = t.cnes
                  AND c.competencia = t.competencia
                  AND ROW(
                      c.codigo_ibge_7,
                      c.municipio_ibge,
                      c.metodo_vinculo,
                      c.status_vinculo
                  ) IS DISTINCT FROM ROW(
                      t.codigo_ibge_7,
                      t.municipio_ibge,
                      t.metodo_vinculo,
                      t.status_vinculo
                  )
            """)

            atualizados = cursor.rowcount

            cursor.execute("""
                SELECT COUNT(*)
                FROM silver.cnes_leitos AS c
                JOIN tmp_vinculos_cnes AS t
                    ON t.cnes = c.cnes
                   AND t.competencia = c.competencia
                WHERE ROW(
                    c.codigo_ibge_7,
                    c.municipio_ibge,
                    c.metodo_vinculo,
                    c.status_vinculo
                ) IS DISTINCT FROM ROW(
                    t.codigo_ibge_7,
                    t.municipio_ibge,
                    t.metodo_vinculo,
                    t.status_vinculo
                )
            """)

            if cursor.fetchone()[0]:
                raise RuntimeError(
                    "A conferência final falhou; "
                    "as alterações serão desfeitas."
                )

            cursor.execute("""
                SELECT status_vinculo, COUNT(*)
                FROM silver.cnes_leitos
                GROUP BY status_vinculo
                ORDER BY status_vinculo
            """)
            resumo = cursor.fetchall()

    print("Atualização concluída e confirmada!")
    print(f"Registros alterados: {atualizados:,}")

    for status, quantidade in resumo:
        print(f"{status}: {quantidade:,}")


atualizar_vinculos_cnes()

Atualização concluída e confirmada!
Registros alterados: 424,204
VINCULADO_COM_POPULACAO: 424,204
